# Gardner's Nested Allpass FDN

Example for the nested allpass structure: an FDN built by iteratively nesting a feedforward/back allpass around the previous system. SISO (single input, single output).

**Reference:** Gardner, W. G. (1992). *A real-time multichannel room simulator.* J. Acoust. Soc. Am. 92, 1–23.

See also: *Allpass Feedback Delay Networks*, Sebastian J. Schlecht (IEEE Trans. Signal Processing).

— Original MATLAB: Sebastian J. Schlecht, 7 June 2020

In [ ]:
# TODO: add a dsp diagram of the nested allpass FDN to the intro

## Setup

In [ ]:
import numpy as np
import pyFDN
from IPython.display import Audio, display

np.random.seed(42)

## Build nested allpass FDN

Use a vector of gains **g** (one per nesting stage). Delays are powers of two: **m = 2^(0:N-1)**.

In [ ]:
N = 6
g = np.array([0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
delays = np.random.randint(200, 1000, size=N)

A, B, C, D = pyFDN.nested_allpass(g)
print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C.shape)
print("D shape:", D.shape)

## Plot the system matrix

In [ ]:
fig = pyFDN.plot_system_matrix(A, B, C, D)
fig.show()

## Test: uniallpass

Check that the FDN is uniallpass (lossless with diagonal Lyapunov matrix).

In [ ]:
is_a, P = pyFDN.is_uniallpass(A, B, C, D)
assert is_a, "Expected uniallpass"
print("Uniallpass: OK")

## Impulse response

Render the IR and plot (SISO: single channel).

In [ ]:
Fs = 48000
ir_len = 4 * Fs  # 4 seconds
impulse_response = pyFDN.dss_to_impz(ir_len, delays, A, B, C, D).squeeze()
# Shape: (ir_len, n_out, n_in) -> (ir_len, ) for SISO

## Impulse response plot and matrix view

For SISO we have one channel; plot_impulse_response_matrix shows a single subplot.

In [ ]:
import matplotlib.pyplot as plt
plt.plot(
    np.arange(ir_len) / Fs,
    pyFDN.mulaw_encode(impulse_response),
)
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [mu-law]")
plt.show()

## Spectrogram (one channel)

In [ ]:
channel_ir = np.asarray(impulse_response).squeeze()
fig = pyFDN.plot_spectrogram(channel_ir, Fs, title="Nested allpass FDN — spectrogram")
fig.show()

display(Audio(channel_ir, rate=Fs))